# `elo` -- Chronological Elo Rating System

Implements a classic chess-style Elo rating system adapted to IPL team strength. The core idea: every team starts at a neutral rating (1500), and after each match the winner takes rating points from the loser -- the *amount* transferred depends on how surprising the result was (beating a much-stronger team transfers more points than beating an equal).

**Critical property: no lookahead.** Ratings are updated strictly in match order, and the rating recorded for a match is always the rating *before* that match was played. This is what makes Elo usable as a pre-match feature without leaking future information -- verified by `tests/test_elo.py::test_no_mutation_of_input`.

In [1]:
import pandas as pd

## `compute_elo()`

**Inputs:** `match_df` must already be sorted chronologically (by `match_id`, which increases with date in this dataset).

**Algorithm** (standard Elo, K=32, init=1500):
1. For each match, look up team1's and team2's *current* ratings    (defaulting to 1500 if a team hasn't played yet).
2. Compute team1's *expected* win probability from the rating gap:    `exp1 = 1 / (1 + 10^((e2-e1)/400))` -- this is the standard    logistic Elo expectation curve (400-point gap == ~10:1 odds).
3. Record the **pre-match** ratings for this row (this is what    becomes the `elo1`/`elo2` feature columns -- the actual result    of this match hasn't affected these numbers yet).
4. *After* recording, update both teams' ratings based on the    actual outcome vs. the expectation: `K * (actual - expected)`.    A team gets a bigger boost from an *unexpected* win than an    expected one.

**Outputs:**
- `match_df` with three new columns: `elo1`, `elo2` (pre-match   ratings) and `elo_diff` (`elo1 - elo2`, the feature actually fed   into the pre-match model).
- `final_elo`: the end-of-dataset rating for every team, used to   seed the 2026 external-holdout walk-forward evaluation in   `pipeline.py::evaluate_2026_pre_match` (so 2026 predictions   start from each team's true up-to-date strength, not a reset   to 1500).

In [2]:
def compute_elo(
    match_df: pd.DataFrame,
    K: int = 32,
    init: int = 1500,
) -> tuple[pd.DataFrame, dict[str, float]]:
    """
    Compute pre-match Elo ratings for every row in match_df.

    match_df must be sorted chronologically before calling (by match_id or date).
    Each team starts at `init`; ratings are updated using the standard
    Elo update rule after each match.

    Returns
    -------
    match_df_with_elo : pd.DataFrame
        Input DataFrame with three columns added:
        ``elo1`` (team1 pre-match rating), ``elo2`` (team2 pre-match rating),
        ``elo_diff`` (elo1 - elo2).
    final_elo : dict[str, float]
        End-of-season ratings for every team, for use in walk-forward
        evaluation on the 2026 external test set.
    """
    elo: dict[str, float] = {}
    elo1_list, elo2_list = [], []

    for _, row in match_df.iterrows():
        t1, t2 = row["team1"], row["team2"]
        e1 = elo.get(t1, float(init))
        e2 = elo.get(t2, float(init))
        exp1 = 1.0 / (1.0 + 10.0 ** ((e2 - e1) / 400.0))
        elo1_list.append(e1)
        elo2_list.append(e2)
        outcome = row["team1_win"]
        elo[t1] = e1 + K * (outcome - exp1)
        elo[t2] = e2 + K * ((1 - outcome) - (1 - exp1))

    out = match_df.copy()
    out["elo1"] = elo1_list
    out["elo2"] = elo2_list
    out["elo_diff"] = out["elo1"] - out["elo2"]
    return out, elo